# AfghanBazaar - generate the 30 sample product images (free Colab GPU)

Creates one image per product with **Stable Diffusion XL** (free to download, no account or token needed),
saves each under its exact file name, resizes to **1600 x 1600 JPEG**, validates them and builds a ZIP to download.

**How to run (about 5 clicks):**
1. Menu **Runtime > Change runtime type > T4 GPU > Save**.
2. Menu **Runtime > Run all**. First run downloads the model (~7 GB, a few minutes), then makes the images (~15-25 min).
3. When the last cell finishes, your browser downloads `afghanbazaar-product-images.zip`.
4. Look at the contact sheet. To redo a bad image, add its file name to `FORCE` in *Settings* and run the cells from *Settings* down again.

Nothing here uploads to Supabase and no credentials are used. Free GPU time is not guaranteed by Google: if you hit a limit,
wait and run again; finished images are kept and skipped (resume). Set `USE_DRIVE = True` in *Settings* to also keep progress in
your Google Drive if the session disconnects.

In [ ]:
#@title 1. Check the GPU
import subprocess
_out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if _out.returncode != 0:
    raise SystemExit("No GPU found. Use the menu: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all.")
print("GPU:", _out.stdout.strip())

In [ ]:
#@title 2. Install libraries (about 1 minute)
%pip install -q "diffusers>=0.30" transformers accelerate safetensors pillow tqdm

In [ ]:
#@title 3. Settings
import hashlib, io, json, os, re, shutil, time, zipfile
from pathlib import Path
from PIL import Image, ImageStat, ImageDraw
from tqdm.auto import tqdm

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"   # free, public, no token needed
GEN_SIZE = 1024        # SDXL's native size
OUT_SIZE = 1600        # final square size (images are upscaled from 1024)
STEPS = 30
GUIDANCE = 6.5
MAX_ATTEMPTS = 4       # tries per image (each uses a new random seed)
ONLY = None            # e.g. [1, 5, 9] = only these product numbers (1-30)
FORCE = []             # e.g. ["07-afghan-baluchi-prayer-rug.jpg"] = redo these files
USE_DRIVE = False      # True = keep progress in Google Drive (asks you to allow access)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
OUT_DIR = Path(os.environ.get("PRODUCT_IMAGES_DIR") or ("/content/drive/MyDrive/AfghanBazaar-product-images" if USE_DRIVE else "/content/product-images"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUT_DIR.parent / "afghanbazaar-product-images.zip"
print("Images are saved in:", OUT_DIR)

In [ ]:
#@title 4. Load the 30 prompts (copy of seed-assets/product-images/manifest.json)
MANIFEST_JSON = r"""{
  "negativePrompt": "cartoon, illustration, 3d render, distorted objects, extra items, text, lettering, logo, watermark, blurry, oversaturated, fake label",
  "compactStyle": "professional e-commerce product photo, centered, plain light grey seamless background, soft natural window light, realistic textures, true colours, sharp focus",
  "items": [
    {
      "productId": "00000000-0000-4000-8000-0000000a0001",
      "name": "Herat Saffron Threads (5 g)",
      "category": "saffron-spices",
      "file": "01-herat-saffron-5g.jpg",
      "subject": "a small round metal tin with the lid beside it, filled with deep-red whole saffron threads, a few loose threads on a light wooden surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0001/01-herat-saffron-5g.jpg",
      "prompt": "a small round metal tin with the lid beside it, filled with deep-red whole saffron threads, a few loose threads on a light wooden surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0002",
      "name": "Hand-Knotted Afghan Wool Rug (90 x 60 cm)",
      "category": "carpets-rugs",
      "file": "02-afghan-wool-rug-90x60.jpg",
      "subject": "a small hand-knotted wool rug with a traditional red and navy geometric medallion pattern, one corner gently folded to show the pile, photographed from above at a slight angle",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0002/02-afghan-wool-rug-90x60.jpg",
      "prompt": "a small hand-knotted wool rug with a traditional red and navy geometric medallion pattern, one corner gently folded to show the pile, photographed from above at a slight angle. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0003",
      "name": "Lapis Lazuli Pendant with Silver Setting",
      "category": "jewelry-handicrafts",
      "file": "03-lapis-lazuli-pendant.jpg",
      "subject": "a polished deep-blue lapis lazuli pendant with fine golden pyrite flecks in a silver-tone setting, on a thin chain curled on a pale stone surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0003/03-lapis-lazuli-pendant.jpg",
      "prompt": "a polished deep-blue lapis lazuli pendant with fine golden pyrite flecks in a silver-tone setting, on a thin chain curled on a pale stone surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0004",
      "name": "Darjeeling First Flush Black Tea (250 g)",
      "category": "other",
      "file": "04-darjeeling-first-flush-tea.jpg",
      "subject": "a plain unlabeled kraft paper pouch beside a small white cup of pale golden tea and a wooden scoop of loose whole tea leaves",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0004/04-darjeeling-first-flush-tea.jpg",
      "prompt": "a plain unlabeled kraft paper pouch beside a small white cup of pale golden tea and a wooden scoop of loose whole tea leaves. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0005",
      "name": "Kashmiri Embroidered Wool-Blend Shawl",
      "category": "traditional-clothing",
      "file": "05-kashmiri-embroidered-shawl.jpg",
      "subject": "a neatly folded muted-teal wool-blend shawl with detailed floral needle embroidery along the border and fringe visible",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0005/05-kashmiri-embroidered-shawl.jpg",
      "prompt": "a neatly folded muted-teal wool-blend shawl with detailed floral needle embroidery along the border and fringe visible. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0006",
      "name": "Jaipur Block-Print Cotton Bedsheet Set",
      "category": "home-decoration",
      "file": "06-jaipur-block-print-bedsheet.jpg",
      "subject": "a folded indigo-and-white hand block-printed cotton bedsheet with two matching pillow covers stacked on top, floral repeat pattern visible",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0006/06-jaipur-block-print-bedsheet.jpg",
      "prompt": "a folded indigo-and-white hand block-printed cotton bedsheet with two matching pillow covers stacked on top, floral repeat pattern visible. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0007",
      "name": "Afghan Baluchi Prayer Rug (Wool, 130 x 80 cm)",
      "category": "carpets-rugs",
      "file": "07-afghan-baluchi-prayer-rug.jpg",
      "subject": "a hand-woven wool prayer rug in deep burgundy, charcoal and ivory with an arch-shaped central design, laid flat and photographed from above",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0007/07-afghan-baluchi-prayer-rug.jpg",
      "prompt": "a hand-woven wool prayer rug in deep burgundy, charcoal and ivory with an arch-shaped central design, laid flat and photographed from above. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0008",
      "name": "Afghan Kilim Flat-Weave Wool Rug (120 x 80 cm)",
      "category": "carpets-rugs",
      "file": "08-afghan-kilim-flatweave-rug.jpg",
      "subject": "a flat-woven wool kilim rug with bold geometric bands in rust, indigo and cream, loosely rolled at one end to show the weave",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0008/08-afghan-kilim-flatweave-rug.jpg",
      "prompt": "a flat-woven wool kilim rug with bold geometric bands in rust, indigo and cream, loosely rolled at one end to show the weave. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0009",
      "name": "Kashmiri Hand-Knotted Silk-Blend Rug (90 x 60 cm)",
      "category": "carpets-rugs",
      "file": "09-kashmiri-silk-blend-rug.jpg",
      "subject": "a fine hand-knotted silk-blend rug with a floral medallion in rose, sage and gold tones, soft sheen on the pile, photographed from above at a slight angle",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0009/09-kashmiri-silk-blend-rug.jpg",
      "prompt": "a fine hand-knotted silk-blend rug with a floral medallion in rose, sage and gold tones, soft sheen on the pile, photographed from above at a slight angle. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0010",
      "name": "Jaipur Hand-Woven Cotton Dhurrie (150 x 90 cm)",
      "category": "carpets-rugs",
      "file": "10-jaipur-cotton-dhurrie.jpg",
      "subject": "a flat-weave cotton dhurrie with mustard and indigo stripes and tasselled ends, laid flat on a pale floor",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0010/10-jaipur-cotton-dhurrie.jpg",
      "prompt": "a flat-weave cotton dhurrie with mustard and indigo stripes and tasselled ends, laid flat on a pale floor. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0011",
      "name": "Afghan Green Raisins - Kishmish (500 g)",
      "category": "dried-fruits-nuts",
      "file": "11-afghan-green-raisins-500g.jpg",
      "subject": "a rustic wooden bowl heaped with green raisins, a small pile of raisins spilling onto the surface, no packaging text",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0011/11-afghan-green-raisins-500g.jpg",
      "prompt": "a rustic wooden bowl heaped with green raisins, a small pile of raisins spilling onto the surface, no packaging text. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0012",
      "name": "Afghan Pistachios, In-Shell (500 g)",
      "category": "dried-fruits-nuts",
      "file": "12-afghan-pistachios-500g.jpg",
      "subject": "a shallow woven basket of in-shell pistachios, a few cracked open showing green kernels, on a light wooden surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0012/12-afghan-pistachios-500g.jpg",
      "prompt": "a shallow woven basket of in-shell pistachios, a few cracked open showing green kernels, on a light wooden surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0013",
      "name": "Indian Whole Cashews W240 (500 g)",
      "category": "dried-fruits-nuts",
      "file": "13-indian-whole-cashews-500g.jpg",
      "subject": "a white ceramic bowl filled with whole pale cashew kernels, some scattered in front, on a light surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0013/13-indian-whole-cashews-500g.jpg",
      "prompt": "a white ceramic bowl filled with whole pale cashew kernels, some scattered in front, on a light surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0014",
      "name": "Kashmiri Mamra-Style Almonds (250 g)",
      "category": "dried-fruits-nuts",
      "file": "14-kashmiri-mamra-almonds-250g.jpg",
      "subject": "a small brass bowl of whole brown almonds, a handful scattered beside it on a light wooden surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0014/14-kashmiri-mamra-almonds-250g.jpg",
      "prompt": "a small brass bowl of whole brown almonds, a handful scattered beside it on a light wooden surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0015",
      "name": "Afghan Black Cumin Seeds - Zira Siyah (100 g)",
      "category": "saffron-spices",
      "file": "15-afghan-black-cumin-100g.jpg",
      "subject": "a small clear glass jar with a plain lid filled with thin dark cumin seeds, and a small wooden spoon of seeds beside it",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0015/15-afghan-black-cumin-100g.jpg",
      "prompt": "a small clear glass jar with a plain lid filled with thin dark cumin seeds, and a small wooden spoon of seeds beside it. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0016",
      "name": "Kashmiri Mongra Saffron (1 g)",
      "category": "saffron-spices",
      "file": "16-kashmiri-mongra-saffron-1g.jpg",
      "subject": "a tiny clear glass vial with a cork stopper holding bright red saffron threads, a few threads beside it on a pale ceramic dish",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0016/16-kashmiri-mongra-saffron-1g.jpg",
      "prompt": "a tiny clear glass vial with a cork stopper holding bright red saffron threads, a few threads beside it on a pale ceramic dish. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0017",
      "name": "Kerala Green Cardamom Pods (100 g)",
      "category": "saffron-spices",
      "file": "17-kerala-green-cardamom-100g.jpg",
      "subject": "a small wooden bowl heaped with whole green cardamom pods, a few pods scattered in front on a light surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0017/17-kerala-green-cardamom-100g.jpg",
      "prompt": "a small wooden bowl heaped with whole green cardamom pods, a few pods scattered in front on a light surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0018",
      "name": "Afghan Kuchi Tribal Statement Necklace (Silver-Tone)",
      "category": "jewelry-handicrafts",
      "file": "18-kuchi-tribal-necklace.jpg",
      "subject": "a handcrafted silver-tone tribal statement necklace with engraved discs, small coloured glass accents and dangling pendants, arranged in a curve on a neutral surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0018/18-kuchi-tribal-necklace.jpg",
      "prompt": "a handcrafted silver-tone tribal statement necklace with engraved discs, small coloured glass accents and dangling pendants, arranged in a curve on a neutral surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0019",
      "name": "Lapis Lazuli Bead Bracelet",
      "category": "jewelry-handicrafts",
      "file": "19-lapis-lazuli-bead-bracelet.jpg",
      "subject": "a bracelet of round polished deep-blue lapis lazuli beads with golden flecks, coiled on a pale stone surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0019/19-lapis-lazuli-bead-bracelet.jpg",
      "prompt": "a bracelet of round polished deep-blue lapis lazuli beads with golden flecks, coiled on a pale stone surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0020",
      "name": "Jaipur Meenakari Enamel Jhumka Earrings",
      "category": "jewelry-handicrafts",
      "file": "20-meenakari-jhumka-earrings.jpg",
      "subject": "a pair of dome-shaped gold-tone jhumka earrings with hand-painted green, pink and blue enamel and tiny hanging beads, on a neutral surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0020/20-meenakari-jhumka-earrings.jpg",
      "prompt": "a pair of dome-shaped gold-tone jhumka earrings with hand-painted green, pink and blue enamel and tiny hanging beads, on a neutral surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0021",
      "name": "Channapatna Lacquered Wooden Toy Set (5 pieces)",
      "category": "jewelry-handicrafts",
      "file": "21-channapatna-wooden-toy-set.jpg",
      "subject": "five hand-turned lacquered wooden toys in bright red, yellow, green, blue and orange, a spinning top, small figures and a ring stack, arranged neatly on a light surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0021/21-channapatna-wooden-toy-set.jpg",
      "prompt": "five hand-turned lacquered wooden toys in bright red, yellow, green, blue and orange, a spinning top, small figures and a ring stack, arranged neatly on a light surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0022",
      "name": "Afghan Hand-Embroidered Cotton Dress (Kuchi-Style)",
      "category": "traditional-clothing",
      "file": "22-afghan-embroidered-dress.jpg",
      "subject": "a red and black cotton dress with dense hand embroidery and small mirror-work on the bodice, laid out flat and photographed from above, no person",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0022/22-afghan-embroidered-dress.jpg",
      "prompt": "a red and black cotton dress with dense hand embroidery and small mirror-work on the bodice, laid out flat and photographed from above, no person. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0023",
      "name": "Afghan Pakol Wool Cap",
      "category": "traditional-clothing",
      "file": "23-afghan-pakol-wool-cap.jpg",
      "subject": "a traditional round rolled-brim wool pakol cap in oatmeal-brown, shown from a slight three-quarter angle on a neutral surface, no person",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0023/23-afghan-pakol-wool-cap.jpg",
      "prompt": "a traditional round rolled-brim wool pakol cap in oatmeal-brown, shown from a slight three-quarter angle on a neutral surface, no person. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0024",
      "name": "Banarasi Silk-Blend Dupatta",
      "category": "traditional-clothing",
      "file": "24-banarasi-silk-blend-dupatta.jpg",
      "subject": "a folded deep-maroon silk-blend dupatta with a woven gold-tone border and floral motifs, one end draped to show the pattern",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0024/24-banarasi-silk-blend-dupatta.jpg",
      "prompt": "a folded deep-maroon silk-blend dupatta with a woven gold-tone border and floral motifs, one end draped to show the pattern. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0025",
      "name": "Istalif-Style Turquoise Glazed Ceramic Plate (28 cm)",
      "category": "home-decoration",
      "file": "25-istalif-turquoise-ceramic-plate.jpg",
      "subject": "a hand-thrown ceramic plate with glossy turquoise glaze and dark floral detail, standing at a slight angle against a plain backdrop on a small wooden stand",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0025/25-istalif-turquoise-ceramic-plate.jpg",
      "prompt": "a hand-thrown ceramic plate with glossy turquoise glaze and dark floral detail, standing at a slight angle against a plain backdrop on a small wooden stand. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0026",
      "name": "Afghan Felted Wool Cushion Cover (45 x 45 cm)",
      "category": "home-decoration",
      "file": "26-afghan-felt-cushion-cover.jpg",
      "subject": "a square felted wool cushion with a bold geometric pattern in cream, rust and charcoal, standing upright on a neutral surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0026/26-afghan-felt-cushion-cover.jpg",
      "prompt": "a square felted wool cushion with a bold geometric pattern in cream, rust and charcoal, standing upright on a neutral surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0027",
      "name": "Jaipur Blue Pottery Vase (25 cm)",
      "category": "home-decoration",
      "file": "27-jaipur-blue-pottery-vase.jpg",
      "subject": "a hand-painted blue pottery vase with cobalt and turquoise floral motifs on a white glaze, standing on a plain surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0027/27-jaipur-blue-pottery-vase.jpg",
      "prompt": "a hand-painted blue pottery vase with cobalt and turquoise floral motifs on a white glaze, standing on a plain surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0028",
      "name": "Afghan Wild Mountain Honey (500 g)",
      "category": "other",
      "file": "28-afghan-mountain-honey-500g.jpg",
      "subject": "a plain unlabeled glass jar of thick golden honey with a wooden honey dipper resting beside it, a thin ribbon of honey on the dipper",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0028/28-afghan-mountain-honey-500g.jpg",
      "prompt": "a plain unlabeled glass jar of thick golden honey with a wooden honey dipper resting beside it, a thin ribbon of honey on the dipper. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0029",
      "name": "Afghan Nabat Crystal Rock Sugar (250 g)",
      "category": "other",
      "file": "29-afghan-nabat-rock-sugar-250g.jpg",
      "subject": "a small white ceramic dish of pale amber crystallised rock sugar chunks, a few larger crystals beside it on a light surface",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0029/29-afghan-nabat-rock-sugar-250g.jpg",
      "prompt": "a small white ceramic dish of pale amber crystallised rock sugar chunks, a few larger crystals beside it on a light surface. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    },
    {
      "productId": "00000000-0000-4000-8000-0000000a0030",
      "name": "Kashmiri Kahwa Tea Blend (100 g)",
      "category": "other",
      "file": "30-kashmiri-kahwa-tea-100g.jpg",
      "subject": "a small glass teapot and a small glass cup of pale golden-green tea with a cinnamon stick, a few green tea leaves and cardamom pods on the surface in front",
      "storagePath": "00000000-0000-4000-8000-0000000d3e01/00000000-0000-4000-8000-0000000a0030/30-kashmiri-kahwa-tea-100g.jpg",
      "prompt": "a small glass teapot and a small glass cup of pale golden-green tea with a cinnamon stick, a few green tea leaves and cardamom pods on the surface in front. Professional e-commerce product photograph. Square 1:1 composition with the product centered and generous empty margin (subject fills about 60-65% of the frame), on a clean seamless warm light-grey backdrop. Soft natural window light from the left, gentle realistic shadow, true-to-life colours, realistic material textures, sharp focus. No text, no logos, no labels, no watermark, no people, no hands."
    }
  ]
}"""
manifest = json.loads(MANIFEST_JSON)
_override = Path("/content/manifest.json")          # optional: upload your own manifest.json to /content to use it instead
if _override.exists():
    manifest = json.loads(_override.read_text(encoding="utf-8"))
    print("Using uploaded /content/manifest.json")

FILE_RE = re.compile(r"^[0-9]{2}-[a-z0-9-]+\.jpg$")
ITEMS = manifest["items"]
assert len(ITEMS) == 30, f"expected 30 products, found {len(ITEMS)}"
assert len({i["file"] for i in ITEMS}) == 30, "duplicate file names in manifest"
assert all(FILE_RE.match(i["file"]) for i in ITEMS), "unexpected file name in manifest"
NEGATIVE = manifest["negativePrompt"]
STYLE = manifest["compactStyle"]
for _i in ITEMS:
    _i["final_prompt"] = f'{_i["subject"]}, {STYLE}'
print(f"{len(ITEMS)} prompts loaded. First: {ITEMS[0]['file']}")
print("Example prompt:", ITEMS[0]["final_prompt"])

In [ ]:
#@title 5. Helper functions (checks, resize, progress log)
LOG_PATH = OUT_DIR / "generation-log.json"
log = json.loads(LOG_PATH.read_text()) if LOG_PATH.exists() else []

def check_image(img):
    '''Return a problem description, or None if the raw image looks usable.'''
    if min(img.size) < 512:
        return f"too small {img.size}"
    if max(ImageStat.Stat(img.convert("RGB")).stddev) < 4:
        return "blank or uniform image"
    return None

def to_output_jpeg(img):
    img = img.convert("RGB")
    w, h = img.size
    side = min(w, h)
    img = img.crop(((w - side) // 2, (h - side) // 2, (w - side) // 2 + side, (h - side) // 2 + side))
    img = img.resize((OUT_SIZE, OUT_SIZE), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=92, optimize=True)
    return buf.getvalue()

def file_status(path):
    '''Return (ok, reason) for an image file on disk.'''
    try:
        with Image.open(path) as im:
            im.load()
            if im.format != "JPEG":
                return False, f"not a JPEG ({im.format})"
            if im.size != (OUT_SIZE, OUT_SIZE):
                return False, f"size {im.size}, expected {OUT_SIZE}x{OUT_SIZE}"
            if max(ImageStat.Stat(im.convert("RGB")).stddev) < 4:
                return False, "blank or uniform image"
    except Exception as exc:
        return False, f"unreadable: {exc}"
    if Path(path).stat().st_size > 5 * 1024 * 1024:
        return False, "larger than 5 MB"
    return True, "ok"

def save_log():
    tmp = LOG_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(log, indent=2))
    tmp.replace(LOG_PATH)
print("Ready.")

In [ ]:
#@title 6. Load the model (first run downloads ~7 GB)
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.enable_vae_slicing()

def generate(prompt, negative, seed):
    gen = torch.Generator("cuda").manual_seed(seed)
    return pipe(prompt=prompt, negative_prompt=negative, width=GEN_SIZE, height=GEN_SIZE,
                num_inference_steps=STEPS, guidance_scale=GUIDANCE, generator=gen).images[0]

# SDXL ignores words after ~77 tokens: warn if any prompt is that long.
for _i in ITEMS:
    _n = len(pipe.tokenizer(_i["final_prompt"]).input_ids)
    if _n > 77:
        print(f"WARNING: prompt for {_i['file']} has {_n} tokens; the end will be ignored")
print("Model ready.")

In [ ]:
#@title 7. Generate all images (safe to re-run: finished images are skipped)
def run_generation():
    wanted = [(n, it) for n, it in enumerate(ITEMS, 1) if not ONLY or n in ONLY]
    made, skipped, failed = 0, 0, []
    started = time.time()
    bar = tqdm(wanted, desc="images", unit="img")
    for n, it in bar:
        target = OUT_DIR / it["file"]
        if target.exists() and it["file"] not in FORCE:
            ok, why = file_status(target)
            if ok:
                skipped += 1
                continue
            print(f"  {it['file']} exists but is invalid ({why}); regenerating")
        done = False
        for attempt in range(1, MAX_ATTEMPTS + 1):
            seed = n * 1000 + attempt + (7919 if it["file"] in FORCE else 0)
            try:
                img = generate(it["final_prompt"], NEGATIVE, seed)
                problem = check_image(img)
                if problem:
                    print(f"  {it['file']} attempt {attempt}: rejected ({problem})")
                    continue
                data = to_output_jpeg(img)
                tmp = target.with_suffix(".tmp")
                tmp.write_bytes(data)
                tmp.replace(target)
                log.append({"file": it["file"], "productId": it["productId"], "model": MODEL_ID, "seed": seed,
                            "steps": STEPS, "guidance": GUIDANCE, "bytes": len(data),
                            "sha256": hashlib.sha256(data).hexdigest()})
                save_log()
                made += 1
                done = True
                break
            except Exception as exc:
                print(f"  {it['file']} attempt {attempt}/{MAX_ATTEMPTS} failed: {type(exc).__name__}: {str(exc)[:150]}")
                try:
                    import torch
                    torch.cuda.empty_cache()
                except Exception:
                    pass
                time.sleep(2)
        if not done:
            failed.append(it["file"])
        bar.set_postfix(made=made, skipped=skipped, failed=len(failed))
    mins = (time.time() - started) / 60
    print(f"\nDone in {mins:.1f} min: made={made} skipped={skipped} failed={len(failed)}")
    for f in failed:
        print("FAILED:", f, "(re-run this cell to try again)")
    return failed

FAILED = run_generation()

In [ ]:
#@title 8. Contact sheet - LOOK at every image and check it matches its product
def contact_sheet(cols=6, thumb=260):
    present = [(n, it) for n, it in enumerate(ITEMS, 1) if (OUT_DIR / it["file"]).exists()]
    rows = max(1, (len(present) + cols - 1) // cols)
    sheet = Image.new("RGB", (cols * thumb, rows * (thumb + 28)), "white")
    d = ImageDraw.Draw(sheet)
    for k, (n, it) in enumerate(present):
        with Image.open(OUT_DIR / it["file"]) as im:
            t = im.convert("RGB").resize((thumb, thumb))
        x, y = (k % cols) * thumb, (k // cols) * (thumb + 28)
        sheet.paste(t, (x, y))
        d.text((x + 4, y + thumb + 6), f"{n}. {it['file'][3:34]}", fill="black")
    return sheet

SHEET = contact_sheet()
try:
    from IPython.display import display
    display(SHEET)
except Exception:
    pass
print("Check for: wrong object, garbled text, logos, distorted shapes, people/hands, near-duplicates.")
print("To redo some: put their file names in FORCE (Settings cell), then run Settings and the cells below it again.")

In [ ]:
#@title 9. Validate all 30 files
def dhash(path, size=8):
    with Image.open(path) as im:
        g = im.convert("L").resize((size + 1, size))
    px = list(g.tobytes())
    return sum(1 << i for i, (a, b) in enumerate(((px[r * (size + 1) + c], px[r * (size + 1) + c + 1]) for r in range(size) for c in range(size))) if a > b)

def validate_all():
    problems, sha, hashes = [], {}, {}
    for n, it in enumerate(ITEMS, 1):
        p = OUT_DIR / it["file"]
        if not p.exists():
            problems.append(f"missing: {it['file']}")
            continue
        ok, why = file_status(p)
        if not ok:
            problems.append(f"{it['file']}: {why}")
            continue
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        if digest in sha:
            problems.append(f"{it['file']} is byte-identical to {sha[digest]}")
        sha[digest] = it["file"]
        hashes[it["file"]] = dhash(p)
    names = list(hashes)
    for a in range(len(names)):
        for b in range(a + 1, len(names)):
            if bin(hashes[names[a]] ^ hashes[names[b]]).count("1") <= 3:
                problems.append(f"{names[a]} and {names[b]} look almost identical")
    return problems

PROBLEMS = validate_all()
present = sum((OUT_DIR / i["file"]).exists() for i in ITEMS)
print(f"{present}/30 images present")
if PROBLEMS:
    print("PROBLEMS:")
    for p in PROBLEMS:
        print(" -", p)
else:
    print("ALL 30 images are valid, correctly named, 1600x1600 JPEG and distinct.")

In [ ]:
#@title 10. Build the ZIP
def build_zip():
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        for it in ITEMS:
            p = OUT_DIR / it["file"]
            if p.exists():
                z.write(p, arcname=it["file"])
        if LOG_PATH.exists():
            z.write(LOG_PATH, arcname="generation-log.json")
    return ZIP_PATH

build_zip()
with zipfile.ZipFile(ZIP_PATH) as z:
    names = [n for n in z.namelist() if n.endswith(".jpg")]
print(f"{ZIP_PATH.name}: {len(names)} images, {ZIP_PATH.stat().st_size // 1024 // 1024} MB")
if PROBLEMS:
    print("NOTE: the validation cell reported problems - fix those before using these images.")

In [ ]:
#@title 11. Download the ZIP
from google.colab import files
files.download(str(ZIP_PATH))

## After the download
Unzip into `seed-assets/product-images/` in the project (the 30 `.jpg` files go directly in that folder), then:
`node scripts/seed-catalog/validate.mjs --require-images`. Tell Claude when the files are in place, and the images get reviewed
and uploaded to the private bucket with the existing upload script. Review each image yourself first.